# Week 50, Day 1: Ten rules for modelling behaviour: simulation and recovery

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w050/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: One task, three hypotheses

Complete choice_prob(model, par, Q, t, a, rw), the probability of choosing option 1 on trial t under each model, then simulate each model once with typical parameters and report how often it chose the arm that was better at the time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
r = np.random.default_rng(2019)
T = 200; mu = np.array([0.2, 0.8])

def choice_prob(model, par, Q, t, a, rw):
    if model == "ql":
        ...
    if model == "wsls":
        if t == 0:
            return 0.5
        stay = 1 - par[0] / 2
        return stay if ((a[t - 1] == 1) == (rw[t - 1] == 1)) else 1 - stay
    return par[0]

def simulate(model, par):
    a, rw, Q = np.zeros(T, int), np.zeros(T), np.zeros(2)
    for t in range(T):
        a[t] = r.random() < choice_prob(model, par, Q, t, a, rw)
        rw[t] = r.random() < (mu if t < T // 2 else mu[::-1])[a[t]]
        if model == "ql":
            Q[a[t]] += par[0] * (rw[t] - Q[a[t]])
    return a, rw

typical = {"ql": [0.3, 5.0], "wsls": [0.2], "bias": [0.5]}
better = np.r_[np.ones(T // 2), np.zeros(T - T // 2)]
summaries = {m: (simulate(m, p)[0] == better).mean() for m, p in typical.items()}
print({m: round(float(v), 2) for m, v in summaries.items()}, "fraction of choices of the currently better arm")

In [ ]:
import numpy as np
def test_ql():
    "the learning model chooses the currently better arm more than 70% of the time"
    assert summaries["ql"] > 0.7
def test_bias():
    "the biased random model ignores the arms' values (between 0.35 and 0.65)"
    assert 0.35 < summaries["bias"] < 0.65

test_ql()
test_bias()
print("All checks passed")

### Exercise: Negative log-likelihood and BIC

Write nll(model, par, a, rw) by running choice_prob along the observed choices (updating Q for the learning model), and best_model(a, rw), which fits all three models with L-BFGS-B and returns the name with the lowest BIC and the BIC values.

In [ ]:
spec = {"ql": ([0.5, 3.0], [(0.01, 1), (0, 20)]), "wsls": ([0.3], [(0.01, 1)]), "bias": ([0.5], [(0.01, 0.99)])}

def nll(model, par, a, rw):
    Q = np.zeros(2); L = 0.0
    for t in range(T):
        p1 = min(max(choice_prob(model, par, Q, t, a, rw), 1e-6), 1 - 1e-6)
        ...
    return L

def best_model(a, rw):
    bic = {}
    for m, (x0, bounds) in spec.items():
        res = minimize(lambda p: nll(m, p, a, rw), x0, method="L-BFGS-B", bounds=bounds)
        bic[m] = 2 * res.fun + len(x0) * np.log(T)
    return min(bic, key=bic.get), bic

name, bic = best_model(*simulate("ql", [0.3, 6.0]))
print("selected:", name, {m: round(float(v), 1) for m, v in bic.items()})

In [ ]:
import numpy as np
def test_self():
    "on data from Q-learning with a strong beta, BIC selects Q-learning"
    a, rw = simulate("ql", [0.3, 6.0])
    assert best_model(a, rw)[0] == "ql"

test_self()
print("All checks passed")

### Exercise: Can the models be told apart?

For each generating model, simulate 15 participants with parameters drawn from plausible ranges (alpha 0.1 to 0.6 and beta 2 to 8; epsilon 0.05 to 0.3; p 0.3 to 0.7), select the best model for each, and fill a 3 by 3 matrix of proportions.

In [ ]:
draw = {"ql": lambda: [r.uniform(0.1, 0.6), r.uniform(2, 8)], "wsls": lambda: [r.uniform(0.05, 0.3)], "bias": lambda: [r.uniform(0.3, 0.7)]}
names = list(spec); C = np.zeros((3, 3))
for i, m in enumerate(names):
    for _ in range(15):
        ...
C /= 15
print("rows = generating model, columns = selected model", names); print(np.round(C, 2))
plt.imshow(C, vmin=0, vmax=1, cmap="Blues"); plt.xticks(range(3), names); plt.yticks(range(3), names); plt.colorbar(); plt.show()

In [ ]:
import numpy as np
def test_rows():
    "each row of the confusion matrix sums to 1"
    assert np.allclose(C.sum(1), 1)
def test_diag():
    "every model is recovered at least 60% of the time"
    assert (np.diag(C) >= 0.6).all()

test_rows()
test_diag()
print("All checks passed")